# Parte C — Cliente HTTP con `requests`

**Antes de arrancar:** en una terminal aparte (con el venv activado), corré tu API:

```bash
uvicorn main:app --reload
```

Dejala corriendo mientras trabajás en este notebook — acá solo hacemos de **cliente**, pegándole a esa API desde afuera.

In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

## 🟡 C1 — GET desde un script
Escribí un script aparte que use `requests` para pegarle al `GET /libros` y que imprima la respuesta.

**Listo cuando:** el script imprime lo mismo que viste en `/docs`.

In [4]:
import requests
 
response = requests.get('http://127.0.0.1:8000/libros',timeout=5)
print(f"status code: {response.status_code}")  # 200

lista = response.json()  # Response body as dic
for i in lista:
    print(i)

status code: 200
{'titulo': 'cien años de soledad', 'autor': 'Gabriel Garcia Marquez', 'paginas': 420, 'disponible': True}
{'titulo': 'rebelion en la granja', 'autor': 'George Orwell', 'paginas': 190, 'disponible': False}
{'titulo': 'fahrenheit 451', 'autor': 'ray bradbury', 'paginas': 172, 'disponible': True, 'editorial': {'nombre': 'minotauro', 'pais': 'españa'}}


## 🟡 C2 — POST desde un script
Desde el script, hacé un `POST /libros` con un libro nuevo. Después hacé un GET y confirmá que se agregó.

**Listo cuando:** cargaste un dato por script y lo recuperás por script.

In [10]:
import requests

URL = "http://127.0.0.1:8000/libros"

payload = {
    "titulo": "el camino de los reyes",
    "autor": "Brandon sanderson",
    "paginas": 1200,
    "disponible": False
}

response = requests.post(URL, json=payload)
print(f"status code: {response.status_code}")
print(response.json())

status code: 201
{'titulo': 'el camino de los reyes', 'autor': 'Brandon sanderson', 'paginas': 1200, 'disponible': False, 'editorial': None, 'costo_interno': 0.0}


## 🔴 C3 — "En `/docs` andaba y acá no"
Es muy probable que el POST del script falle aunque en `/docs` funcione. Encontrá por qué (pista: cómo se manda el body, el header de JSON, la URL/puerto exactos).

**Listo cuando:** el script anda *y sabés explicar* qué le faltaba comparado con `/docs`.

### utilice "response = requests.post(URL, json=payload)" donde 'payload' es un dic con el cuerpo del objeto LIBRO del la parte-b, y ese parte 'json=payload' hace que, no sea falta declarar el header 'Content-Type: application/json', ya que se te lo declara por ti, siendo solo necesario crear el BODY.

## 🔴 C4 — Leer el código de estado en el cliente
Hacé que tu script imprima distinto según la respuesta: "ok" si fue 200/201, "dato inválido" si fue 422, "no existe" si fue 404.

**Listo cuando:** el cliente reacciona al código, no solo al cuerpo.

rta: use la funcion 'resonse.status_code'que devuelve el estado de la peticion.

## 🟡 C5 — PUT y DELETE desde el cliente
Sumá al script funciones que hagan `PUT` (reemplazar un libro) y `DELETE` (borrarlo), usando lo que armaste en B6/B7.

**Listo cuando:** tu script puede crear, listar, reemplazar y borrar un libro sin tocar `/docs`.

In [ ]:
import requests
libro="el camino de los reyes"
url = f"http://127.0.0.1:8000/libros/{libro}"

payload = {
    "titulo": "frankeinstain",
    "autor": "Mary Shelley",
    "paginas": 288,
    "disponible": False
}
response = requests.put(url, json=payload)
print(f"status code: {response.status_code}")
print(response.json())

In [12]:
import requests
libro="el camino de los reyes"
url = f"http://127.0.0.1:8000/libros/{libro}"

payload = {
    "titulo": "frankeinstain",
    "autor": "Mary Shelley",
    "paginas": 288,
    "disponible": False
}
response = requests.delete(url)
print(f"status code: {response.status_code}")
print(response.json())

status code: 404
{'detail': 'Libro no encontrado'}


## 🔴 C6 — Cuando la API no está levantada
Apagá el servidor de FastAPI y corré tu script cliente. ¿Qué excepción de `requests` salta? Envolvé el pedido en un `try/except` que imprima un mensaje entendible en vez de que el script explote con un traceback.

**Listo cuando:** con la API apagada, tu script avisa "no se pudo conectar" en vez de crashear, y sabés el nombre de la excepción que atrapaste.

In [1]:
import requests

url = "http://127.0.0.1:8000"

try:
    respuesta = requests.get(url, timeout=5) # Timeout de 5 segundos
    respuesta.raise_for_status() # Verificar errores HTTP
    print("Petición exitosa.")

except requests.exceptions.ConnectionError:
    print("Error: No se pudo conectar al servidor.")
except requests.exceptions.Timeout:
    print("Error: La petición tardó demasiado tiempo.")
except requests.exceptions.HTTPError as e:
    print(f"Error HTTP: {e}")
except requests.exceptions.RequestException as e:
    print(f"Ocurrió un error general en la petición: {e}")

Error: No se pudo conectar al servidor.


## 🔴 C7 — Timeout
Investigá el parámetro `timeout` de `requests.get`/`requests.post`. ¿Qué problema previene? Agregalo a tus pedidos con un valor razonable (ej. 5 segundos).

Después, provocá el timeout *de verdad*: contra tu propia API en `localhost`, un valor como `timeout=0.001` puede no alcanzar para nada (¡la respuesta local a veces llega en menos de un milisegundo!) y el ejercicio "no funciona" sin que hayas hecho nada mal. Para ver la excepción de forma confiable, apuntá a una IP que nadie contesta, ej. `requests.get("http://10.255.255.1/", timeout=1)`: el pedido se queda esperando una respuesta que nunca llega, así que el timeout salta siempre.

**Listo cuando:** entendés que sin `timeout` tu script puede quedarse colgado para siempre esperando una respuesta que nunca llega, viste la excepción de timeout al menos una vez, y entendés por qué probarlo contra tu propia API en localhost puede no ser confiable.

What is a Timeout?

A timeout defines how long a program will wait for a server’s response before aborting the connection. It ensures your code doesn’t get stuck when the server is too slow or unreachable.

Why Timeouts Matter

Here’s why every Python developer should always specify a timeout:

Hanging Requests: The program waits forever if the server never responds.
Resource Drain: Open connections consume memory and CPU over time.
Poor UX: Slow responses frustrate users in real-time applications.
Data Inconsistency: Timeouts prevent partial or corrupt data from being processed.

bibliografia: 'https://python-requests.org/python-requests-timeout/'


## 🔴 C8 — Reutilizar la conexión con `Session`
Reescribí tus funciones para usar una `requests.Session()` en vez de llamar a `requests.get`/`.post` sueltos cada vez. Investigá qué ventaja de performance trae cuando hacés muchos pedidos seguidos al mismo servidor.

**Listo cuando:** tu script usa una sola `Session` para todos los pedidos y podés explicar por qué es más eficiente que abrir una conexión nueva por cada uno.

RTA: [C8-session.py](./C8-session.py)